## QAT Validation (Holdout Set)

Evaluate QAT checkpoints (INT8 + INT4) on the 5k holdout-val split, across all seeds.
Same models as `03_2_qat_test`, just on the val set instead of imagenet2.

In [1]:
SKIP_EXISTING = True
OUTPUT_ROOT = "/home/pf4636/code/resnet/quantized_resnets/runs/val/qat"

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
QAT_MOD = PYFILES / "qat_modelopt"
for p in [str(PYFILES), str(QAT_MOD)]:
    if p not in sys.path:
        sys.path.insert(0, p)

In [3]:
import json
import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import modelopt.torch.opt as mto

from torch.utils.data import DataLoader

from src.config import ExperimentConfig, set_seed
from src.data import build_train_holdout_split, build_imagenet_transform
from src.eval import evaluate
from quantize import get_model, restore_modelopt_state

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [4]:
QAT_CKPT_ROOT = PROJECT_ROOT / ".checkpoints" / "qat"
FP32_CKPT_ROOT = PROJECT_ROOT / ".checkpoints"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

QAT_CONFIGS = [
    {"qat_precision": "int8", "input_bits": [8, 4, 2, 1]},
    {"qat_precision": "int4", "input_bits": [8, 4, 2, 1]},
]

checkpoints = {}
skipped = []
for qcfg in QAT_CONFIGS:
    prec = qcfg["qat_precision"]
    for bits in qcfg["input_bits"]:
        run_name = f"{prec}_in{bits}b"
        run_dir = QAT_CKPT_ROOT / run_name
        if not run_dir.exists():
            skipped.append(f"{prec}/in{bits}b (no dir)")
            continue
        for seed_dir in sorted(run_dir.iterdir()):
            if not seed_dir.is_dir() or not seed_dir.name.startswith("seed_"):
                continue
            weights = seed_dir / "qat_modelopt_best.pth"
            mostate = seed_dir / "qat_modelopt_best_mostate.pt"
            seed_num = int(seed_dir.name.split("_")[1])
            fp32_ckpt = FP32_CKPT_ROOT / f"fp32_{bits}bit" / seed_dir.name / "best.pth"
            if not weights.exists() or not mostate.exists() or not fp32_ckpt.exists():
                skipped.append(f"{prec}/in{bits}b/{seed_dir.name}")
                continue
            checkpoints[(prec, bits, seed_dir.name)] = {
                "weights": weights,
                "mostate": mostate,
                "fp32_ckpt": fp32_ckpt,
                "seed": seed_num,
            }

print(f"QAT checkpoints found: {len(checkpoints)}")
for (prec, bits, seed), paths in checkpoints.items():
    print(f"  {prec} / in{bits}b / {seed}")
if skipped:
    print(f"\nSkipped (not complete): {', '.join(skipped)}")

QAT checkpoints found: 24
  int8 / in8b / seed_1
  int8 / in8b / seed_2
  int8 / in8b / seed_42
  int8 / in4b / seed_1
  int8 / in4b / seed_2
  int8 / in4b / seed_42
  int8 / in2b / seed_1
  int8 / in2b / seed_2
  int8 / in2b / seed_42
  int8 / in1b / seed_1
  int8 / in1b / seed_2
  int8 / in1b / seed_42
  int4 / in8b / seed_1
  int4 / in8b / seed_2
  int4 / in8b / seed_42
  int4 / in4b / seed_1
  int4 / in4b / seed_2
  int4 / in4b / seed_42
  int4 / in2b / seed_1
  int4 / in2b / seed_2
  int4 / in2b / seed_42
  int4 / in1b / seed_1
  int4 / in1b / seed_2
  int4 / in1b / seed_42


In [5]:
def load_qat_model(prec: str, bits: int, seed_name: str) -> nn.Module:
    paths = checkpoints[(prec, bits, seed_name)]
    model = get_model(str(paths["fp32_ckpt"]), num_classes=100)
    restore_modelopt_state(model, str(paths["mostate"]))
    ckpt = torch.load(paths["weights"], map_location="cpu")
    state = ckpt["model"] if "model" in ckpt else ckpt
    model.load_state_dict(state)
    model = model.to(DEVICE)
    model.eval()
    return model

def build_val_loader(cfg: ExperimentConfig) -> DataLoader:
    """Holdout-val split (5k samples from training data)."""
    transform = build_imagenet_transform(cfg)
    _, holdout = build_train_holdout_split(
        data_root=cfg.imagenet_path,
        num_classes=cfg.num_classes,
        val_per_class=50,
        seed=42,
        eval_transform=transform,
    )
    return DataLoader(
        holdout,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True,
    )

## QAT Evaluation (All Seeds)

In [6]:
OUT_DIR = Path(OUTPUT_ROOT).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

all_records = []
criterion = nn.CrossEntropyLoss()

for (prec, bits, seed_name), paths in checkpoints.items():
    seed_num = paths["seed"]
    run_id = f"torch_qat_{prec}_b{bits}_cuda"
    result_path = OUT_DIR / seed_name / run_id / "result.json"

    if SKIP_EXISTING and result_path.exists():
        print(f"  {prec}/in{bits}b/{seed_name}: exists, skipping")
        with open(result_path) as f:
            all_records.append(json.load(f))
        continue

    print(f"\n--- QAT {prec.upper()}, input_bits={bits}, {seed_name} ---")
    cfg = ExperimentConfig(
        backend="pytorch",
        device="cuda",
        batch_size=1,
        seed=seed_num,
        num_eval_batches=500,
        input_quant_bits=bits,
        model_precision="fp32",
    )
    set_seed(cfg)

    model = load_qat_model(prec, bits, seed_name)
    val_loader = build_val_loader(cfg)

    t0 = time.perf_counter()
    tracker = evaluate(model, val_loader, cfg, criterion=criterion)
    elapsed = time.perf_counter() - t0

    r = tracker.summary()
    print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

    payload = {
        "status": "ok",
        "run_id": run_id,
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "config_extra": {
            "qat_precision": prec,
            "input_quant_bits": bits,
            "seed": seed_num,
            "split": "holdout_val",
        },
        "results": r,
        "error": None,
        "total_eval_time_sec": round(elapsed, 3),
    }

    result_path.parent.mkdir(parents=True, exist_ok=True)
    with open(result_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

    all_records.append(payload)
    del model
    torch.cuda.empty_cache()

print(f"\n{len(all_records)} runs complete.")


--- QAT INT8, input_bits=8, seed_1 ---
Inserted 107 quantizers


/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of maxpool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(
/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of pool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
Evaluating on 500 batches (first 30 are warmup)...
Loading extension modelopt_cuda_ext...
Loaded extension modelopt_cuda_ext in 0.5 seconds
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 9.16 ms/batch
  Batch [50/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 9.02 ms/batch
  Batch [60/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.64 ms/batch
  Batch [70/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.36 ms/batch
  Batch [80/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.30 ms/batch
  Batch [90/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.48 ms/batch
  Batch [100/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.68 ms/batch
  Batch [110/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.84 ms/batch
  Batch [120/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 9.06 ms/batch
  Batch [130/500] Top-1: 100.00% | Top-5: 100.00% | I

/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of maxpool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(
/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of pool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.61 ms/batch
  Batch [50/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.21 ms/batch
  Batch [60/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.43 ms/batch
  Batch [70/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.65 ms/batch
  Batch [80/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.60 ms/batch
  Batch [90/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.57 ms/batch
  Batch [100/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.50 ms/batch
  Batch [110/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.45 ms/batch
  Batch [120/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.38 ms/batch
  Batch [130/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.29 ms/batch
  Batch [140/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 8.28 ms/ba

## Per-Run Results

In [7]:
rows = []
for rec in all_records:
    r = rec["results"]
    extra = rec.get("config_extra", rec.get("config", {}))
    bits = extra.get("input_quant_bits", rec["config"].get("input_quant_bits", None))
    prec = extra.get("qat_precision", "int8")
    seed = extra.get("seed", 42)
    rows.append({
        "qat_precision": f"qat_{prec}",
        "input_bits": bits,
        "seed": seed,
        "top1": r["top1_acc"],
        "top5": r["top5_acc"],
        "lat_ms": r["infer_ms_avg"],
        "throughput": r.get("throughput_infer_sps", None),
    })

df_all = pd.DataFrame(rows).sort_values(
    ["qat_precision", "input_bits", "seed"], ascending=[True, True, True]
).reset_index(drop=True)
df_all

,qat_precision,input_bits,seed,top1,top5,lat_ms,throughput
0,qat_int4,1,1,90.426,99.362,9.950,100.498
1,qat_int4,1,2,88.936,99.362,9.947,100.531
2,qat_int4,1,42,78.511,95.319,9.710,102.989
3,qat_int4,2,1,92.766,100.000,11.646,85.865
4,qat_int4,2,2,93.191,99.787,11.751,85.096
5,qat_int4,2,42,83.830,97.234,9.145,109.353
6,qat_int4,4,1,98.511,100.000,9.591,104.265
7,qat_int4,4,2,98.298,100.000,9.797,102.069
8,qat_int4,4,42,86.170,98.936,11.067,90.361
9,qat_int4,8,1,98.511,100.000,10.243,97.623


## Averaged Results (mean ± std across seeds)

In [8]:
avg_df = df_all.groupby(["qat_precision", "input_bits"]).agg(
    top1_mean=("top1", "mean"),
    top1_std=("top1", "std"),
    top5_mean=("top5", "mean"),
    top5_std=("top5", "std"),
    infer_ms_mean=("lat_ms", "mean"),
    infer_ms_std=("lat_ms", "std"),
    throughput_mean=("throughput", "mean"),
    n_seeds=("seed", "count"),
).round(3).reset_index()

avg_df = avg_df.sort_values(
    ["qat_precision", "input_bits"], ascending=[True, True]
).reset_index(drop=True)
avg_df

,qat_precision,input_bits,top1_mean,top1_std,top5_mean,top5_std,infer_ms_mean,infer_ms_std,throughput_mean,n_seeds
0,qat_int4,1,85.957,6.492,98.014,2.334,9.869,0.138,101.339,3
1,qat_int4,2,89.929,5.286,99.007,1.539,10.847,1.476,93.438,3
2,qat_int4,4,94.326,7.064,99.645,0.614,10.152,0.799,98.898,3
3,qat_int4,8,94.043,7.373,99.362,1.106,10.305,0.646,97.291,3
4,qat_int8,1,86.525,6.023,97.943,2.275,8.660,0.389,115.621,3
5,qat_int8,2,90.638,4.985,99.362,1.106,8.757,0.730,114.755,3
6,qat_int8,4,94.823,6.204,99.645,0.614,8.480,0.741,118.500,3
7,qat_int8,8,94.752,6.696,99.504,0.860,8.410,0.457,119.137,3


In [9]:
out_path = PROJECT_ROOT / "resultsv2" / "val_runs" / "qat_torch_avg_val.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")

Saved to /home/pf4636/code/resnet/quantized_resnets/resultsv2/val_runs/qat_torch_avg_val.json
